In [ ]:
# @title 1. Clean Environment & Install Dependencies
import os
import shutil
import subprocess
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Work from a stable root (Colab: /content)
ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
os.chdir(ROOT)

# Make sure we don't accumulate nested clones when re-running the notebook
repo_dir = ROOT / "motion-diffusion-model"
if repo_dir.exists() and not (repo_dir / ".git").exists():
    shutil.rmtree(repo_dir)

if not repo_dir.exists():
    print("📂 Cloning repository...")
    subprocess.run(["git", "clone", "https://github.com/GuyTevet/motion-diffusion-model.git", str(repo_dir)], check=True)
else:
    print("✅ Repository already present. Using existing clone.")

os.chdir(repo_dir)

print("📦 Installing Python dependencies (quiet mode)...")
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "scikit-image", "imageio"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imageio==2.33.0", "scikit-image==0.22.0"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/openai/CLIP.git"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "smplx", "chumpy", "trimesh", "moviepy", "gradio", "gdown"], check=True)
subprocess.run(["apt-get", "update"], check=False)
subprocess.run(["apt-get", "install", "-y", "ffmpeg"], check=False)

print("✅ Environment ready.")


In [ ]:
# @title 2. Prepare Minimal HumanML3D Text-Only Dataset (Fixed)
import os
import urllib.request
from pathlib import Path

DATA_ROOT = Path("dataset/HumanML3D")
TEXT_DIR = DATA_ROOT / "texts"
DATA_ROOT.mkdir(parents=True, exist_ok=True)
TEXT_DIR.mkdir(parents=True, exist_ok=True)

print("⬇️ Downloading Mean/Std statistics from official GitHub raw links...")
base_url = "https://github.com/GuyTevet/motion-diffusion-model/raw/main/dataset"
files_map = {
    "Mean.npy": "t2m_mean.npy",
    "Std.npy": "t2m_std.npy",
}
for local_name, remote_name in files_map.items():
    target = DATA_ROOT / local_name
    if target.exists() and target.stat().st_size > 1024:
        print(f"   ✅ {local_name} already present ({target.stat().st_size} bytes)")
        continue
    url = f"{base_url}/{remote_name}"
    print(f"   ☁️ Fetching {remote_name} -> {local_name} ...")
    urllib.request.urlretrieve(url, target)
    print(f"   ✅ Saved to {target} ({target.stat().st_size} bytes)")

# Build a slightly larger text-only split so len(dataset) > 1
splits = {
    "train": [f"colab_sample_{i:04d}" for i in range(1, 5)],
    "val": ["colab_sample_0001", "colab_sample_0002"],
    "test": ["colab_sample_0003", "colab_sample_0004"],
}

def write_split_file(name, ids):
    with open(DATA_ROOT / f"{name}.txt", "w") as f:
        for item in ids:
            f.write(f"{item}\n")

for split_name, id_list in splits.items():
    write_split_file(split_name, id_list)

# Create minimal caption files (caption#tokens#f_tag#to_tag)
text_payloads = {
    "colab_sample_0001": "a person is waving#person waving#0#0\n",
    "colab_sample_0002": "a person is turning around#person turning#0#0\n",
    "colab_sample_0003": "a person is clapping#person clapping#0#0\n",
    "colab_sample_0004": "a person is stepping forward#person stepping#0#0\n",
}
for name, content in text_payloads.items():
    with open(TEXT_DIR / f"{name}.txt", "w") as f:
        f.write(content)

# Touch a dummy motion directory to mirror expected structure (not used in text-only mode)
motion_dir = DATA_ROOT / "new_joint_vecs"
motion_dir.mkdir(exist_ok=True)
print(f"✅ Dataset preparation complete in {DATA_ROOT.resolve()}")


In [ ]:
# @title 3. Download Pretrained Model
import os
import zipfile
import subprocess
from pathlib import Path

file_id = "1PE0PK8e5a5j-7-Xhs5YET5U5pGh0c821"
zip_filename = Path("humanml_trans_enc_512.zip")
model_dir = Path("save")
final_model_path = model_dir / "humanml_trans_enc_512" / "model000200000.pt"
model_dir.mkdir(parents=True, exist_ok=True)

if not final_model_path.exists():
    print("⬇️ Downloading pretrained weights (Google Drive)...")
    subprocess.run(["gdown", "--id", file_id, "-O", str(zip_filename)], check=True)
    print("📂 Unzipping model archive...")
    with zipfile.ZipFile(zip_filename, "r") as zf:
        zf.extractall(model_dir)
    zip_filename.unlink(missing_ok=True)
else:
    print("✅ Model already downloaded.")

print("Model location:", final_model_path)


In [ ]:
# @title 4. Sanity Check: Dataset Loaderfrom pathlib import Pathimport numpy as npfrom data_loaders.humanml.data.dataset import TextOnlyDatasetfrom data_loaders.humanml.utils.get_opt import get_optsplit_file = Path("dataset/HumanML3D/test.txt")print("Test split contains", sum(1 for _ in open(split_file)), "entries")opt = get_opt('./dataset/humanml_opt.txt', device='cpu')mean = np.load('dataset/HumanML3D/Mean.npy')std = np.load('dataset/HumanML3D/Std.npy')dataset = TextOnlyDataset(opt, mean, std, str(split_file))print("TextOnlyDataset length:", len(dataset))

In [ ]:
# @title 5. Run a Quick Generation (no Gradio)import globimport osimport subprocessimport sysfrom pathlib import Pathos.environ.setdefault("XDG_RUNTIME_DIR", "/tmp/runtime-mdm")Path(os.environ["XDG_RUNTIME_DIR"]).mkdir(parents=True, exist_ok=True)# We only keep one prompt to minimize runtimeprompts = ["a person is waving"]print("🚀 Running sampler...")cmd = [    sys.executable, "-m", "sample.generate",    "--model_path", "./model/MDM/mdm_humanml3d.ckpt",    "--dataset", "humanml",    "--model_mode", "classifier_free_guidance",    "--text_prompt", prompts[0],    "--motion_length", "5",    "--result_path", "./results",    "--seed", "0",]print("Command:", " ".join(cmd))subprocess.run(cmd, check=True)print("Generated files:")for fname in glob.glob("results/*.mp4"):    print(" -", fname)